# Qwen2.5-7B 部署全流程

这个 notebook 覆盖 Qwen2.5-7B 的 Transformers、vLLM、SGLang、Ollama、llama.cpp 部署路线。

部署不是只有一个答案。Transformers 适合验证，vLLM 适合服务端吞吐，SGLang 适合复杂生成控制，Ollama 和 llama.cpp 适合本地 GGUF。

## 1. 常用程度排名

| 排名 | 路线 | 主要用途 | 先学理由 |
| --- | --- | --- | --- |
| 1 | Transformers | 本地最小验证 | 最容易检查 tokenizer、chat template、adapter |
| 2 | Ollama | 本地桌面运行 | 命令简单，适合 GGUF 和桌面工具 |
| 3 | vLLM | 服务端 API | OpenAI-compatible，高吞吐 |
| 4 | llama.cpp | CPU/Mac/GGUF 验证 | GGUF 基础生态 |
| 5 | SGLang | 复杂推理流程 | 更强生成控制，学习成本高于 vLLM |

## 2. Transformers 本地推理

Transformers 是部署前的第一步。它可以验证模型能否加载、chat template 是否正确、adapter 是否生效。

关键参数：

- `torch_dtype`：加载精度，常用 `auto`、`bfloat16`、`float16`。
- `device_map`：模型放到哪些设备，单机常用 `auto`。
- `max_new_tokens`：最多生成多少新 token。
- `temperature`：随机性，越高越发散。
- `top_p`：候选 token 累积概率，越低越保守。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

# 会下载并加载模型，确认环境后再执行。
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     device_map="auto",
#     trust_remote_code=True,
# )
# messages = [{"role": "user", "content": "用三句话解释 QLoRA。"}]
# prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9, do_sample=True)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 3. vLLM 服务部署

vLLM 适合把模型部署成 OpenAI-compatible API。它的价值是高吞吐、连续批处理、KV cache 管理和流式输出。

关键参数：

- `--model`：模型名或本地路径。
- `--host`：监听地址。
- `--port`：端口。
- `--dtype`：推理精度。
- `--max-model-len`：最大上下文长度，越大显存占用越高。
- `--gpu-memory-utilization`：允许 vLLM 使用的显存比例。
- `--tensor-parallel-size`：张量并行卡数。

In [ ]:
vllm_command = "vllm serve Qwen/Qwen2.5-7B-Instruct --host 0.0.0.0 --port 8000 --dtype auto --max-model-len 8192 --gpu-memory-utilization 0.90"
print(vllm_command)

In [ ]:
from openai import OpenAI

# vLLM 服务启动后可执行。
# client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     messages=[{"role": "user", "content": "解释 SFT 和 DPO 的区别。"}],
#     temperature=0.7,
# )
# print(response.choices[0].message.content)

## 4. SGLang 部署

SGLang 适合结构化推理和复杂生成流程。它也能提供 OpenAI-compatible API，但更强调生成控制和推理编排。

关键参数：

- `--model-path`：模型名或本地路径。
- `--host`：监听地址。
- `--port`：端口。
- `--tp`：tensor parallel 数量。
- `--context-length`：最大上下文长度。
- `--dtype`：推理精度。

In [ ]:
sglang_command = "python -m sglang.launch_server --model-path Qwen/Qwen2.5-7B-Instruct --host 0.0.0.0 --port 30000 --tp 1 --context-length 8192 --dtype auto"
print(sglang_command)

## 5. Ollama 部署

Ollama 适合本地桌面和命令行使用。Qwen2.5 走 Ollama 时，常见路线是使用 Ollama 模型库里的可用 Qwen 模型，或把自己合并后的模型导出为 GGUF 后写 Modelfile。

关键参数：

- `num_ctx`：上下文长度。
- `temperature`：随机性。
- `top_p`：采样范围。
- `repeat_penalty`：重复惩罚。

In [ ]:
modelfile = """FROM ./qwen2_5_7b-q4_k_m.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0.7
PARAMETER top_p 0.9
SYSTEM 你是一个回答准确、表达清楚的中文助手。
"""

print(modelfile)
print("ollama create qwen2.5-7b-local -f Modelfile")
print("ollama run qwen2.5-7b-local")

## 6. llama.cpp 部署

llama.cpp 是 GGUF 路线的基础工具，适合 CPU、Mac Metal、低资源环境和 GGUF 验证。

关键参数：

- `-m`：GGUF 文件路径。
- `-c`：上下文长度。
- `-ngl`：放到 GPU 的层数。
- `--temp`：采样温度。
- `--top-p`：采样范围。

In [ ]:
llama_cpp_command = "./llama-cli -m ../../outputs/qwen2_5_7b/qwen2_5_7b-q4_k_m.gguf -c 8192 -ngl 999 --temp 0.7 --top-p 0.9 -p '解释 LoRA 和 QLoRA 的区别。'"
print(llama_cpp_command)

## 7. 选择建议

- 先用 Transformers 验证模型和模板。
- 要 API 服务，用 vLLM。
- 要复杂推理编排，用 SGLang。
- 要本地桌面使用，用 Ollama。
- 要验证 GGUF 或低资源运行，用 llama.cpp。